# Feature process
Turn str features into hard-coded
Separate a features per 4h
Merge vital signs and lab results according to the hamd_id

In [ ]:
import pandas as pd
import os
from pathlib import Path
import numpy as np

In [ ]:
raw_dir = r"data/eICU_first24h"
primary_dir = f"data/eICU_first24h_ts1h"

# keep the last n rows according time_resolution, making the total time window 48h
time_resolution = "1h"
if time_resolution == "2h":
    time_resolution_min = 120
    num_timestep = 12
elif time_resolution == "1h":
    time_resolution_min = 60
    num_timestep = 24
elif time_resolution == "30min":
    time_resolution_min = 30
    num_timestep = 48
elif time_resolution == "15min":
    time_resolution_min = 15
    num_timestep = 96
else:
    raise ValueError(f"Unknown time resolution {time_resolution}")

output_dir = f"data/eICU_first24h_ts{time_resolution}"
primary_timeseries_dir = f"{primary_dir}/timeseries"
timeseries_dir = f"{output_dir}/timeseries"
Path(timeseries_dir).mkdir(exist_ok=True, parents=True)

## ICD9 Count and occurrence

In [ ]:
# ICD9 occurrence
diag_path = os.path.join(raw_dir, 'diag.csv')
diag_export_path = os.path.join(timeseries_dir, 'diag.csv')
icd9_export_path = os.path.join(timeseries_dir, 'icd9.csv')

diag_df = pd.read_csv(diag_path, dtype={'icd9_code': str})
diag_df = diag_df[~diag_df['icd9_code'].str.match(r'^[A-Za-z]', na=False)] # Remove ICD10 codes and non-digital ICD9 codes
diag_df['icd9_code'] = diag_df['icd9_code'].apply(lambda x: 'ICD_' + x[:3])

# Map diagnosispriority to numerical values
priority_mapping = {'Primary': 1, 'Major': 2, 'Other': 3}
diag_df['seq_num'] = diag_df['diagnosispriority'].map(priority_mapping)

# Sort by seq_num to prioritize 'Primary', 'Major', 'Other'
diag_df = diag_df.sort_values(by=['seq_num'])

# Drop duplicated rows based on stay_id and icd9_code, keeping the highest priority
diag_df = diag_df.drop_duplicates(subset=['stay_id', 'icd9_code'])

# Sort by stay_id and seq_num
diag_df = diag_df.sort_values(by=['stay_id', 'seq_num'])

# Count stay_id before filtering
stay_id_before = diag_df['stay_id'].nunique()

# Drop stay_id without seq_num=1
valid_stay_ids = diag_df.loc[diag_df['seq_num'] == 1, 'stay_id'].unique()
diag_df = diag_df[diag_df['stay_id'].isin(valid_stay_ids)]

# Count stay_id after filtering
stay_id_after = diag_df['stay_id'].nunique()

# Print stay_id counts
print(f"Number of stay_id before filtering: {stay_id_before}")
print(f"Number of stay_id after filtering: {stay_id_after}")

# Export the result
diag_df.to_csv(diag_export_path, index=False, sep=',')
diag_df.info()

used_stay_ids = diag_df['stay_id'].unique()


In [ ]:
# icd9 count
icd9_counts = diag_df.loc[diag_df['seq_num'] == 1, 'icd9_code'].value_counts()

icd9_df = icd9_counts.reset_index()
icd9_df.columns = ['icd9_code', 'count']

icd9_df.to_csv(icd9_export_path, index=False, sep=',')

icd9_df.info()
used_icd9_list = icd9_df['icd9_code'].tolist()

## Drug

In [ ]:
# 设置路径
drug_path = os.path.join(raw_dir, 'drug.csv')
drug_export_path = os.path.join(timeseries_dir, 'drug.csv')

# 读取 drug.csv
drug_df = pd.read_csv(drug_path)

# 将 drugname 列重命名为 drug
drug_df['drug'] = drug_df['drugname']

# LEFT JOIN diag_df 和 drug_df，通过 stay_id 关联
filtered_drug_df = drug_df.merge(diag_df, left_on='patientunitstayid', right_on='stay_id', how='left')

# 提取 icd9_code 和 drug 两列，并删除重复值
unique_drug_data = filtered_drug_df[['icd9_code', 'drug', 'stay_id']].drop_duplicates()
unique_drug_data.dropna(subset=['icd9_code'], inplace=True)
unique_drug_data = unique_drug_data[unique_drug_data['stay_id'].isin(used_stay_ids)]

# 输出到 CSV 文件
unique_drug_data.to_csv(drug_export_path, index=False)

# 打印数据的信息
unique_drug_data.head()


## Demographics

In [ ]:
demo_path = os.path.join(raw_dir, 'demographics.csv')
demo_export_path = os.path.join(timeseries_dir, 'demographics.csv')

demo_df = pd.read_csv(demo_path, dtype={'icd9_code': str})
demo_df = demo_df.merge(diag_df[diag_df['seq_num']==1], left_on='stay_id', right_on='stay_id', how='right')

demo_df = demo_df[demo_df['stay_id'].isin(used_stay_ids)]
demo_df = demo_df.drop_duplicates(subset=['subject_id', 'stay_id'])
demo_df.drop(columns=['icd9code'], inplace=True)
assert len(demo_df) == len(used_stay_ids)

demo_df.to_csv(demo_export_path,index=False, sep=',')
demo_df.info()

## Vital signs
1. Merge vital.csv and labs_df['bedside glucose'] as the new vital sign df
2. Aggregate the vital signs at 5-min level and get the min, mean, max value, followed by the original paper's VitalPeriodic
3. Set the time resolution to 1 hour

In [ ]:
# merge vital and labs.bedside glucode
vital_path = os.path.join(raw_dir, 'vital.csv')
vital_export_path = os.path.join(timeseries_dir, 'vital.csv')
labs_path = os.path.join(raw_dir, 'labs.csv')

vital_df = pd.read_csv(vital_path)
vital_df = vital_df[vital_df['stay_id'].isin(used_stay_ids)]

# process glucose df
glucose_df = pd.read_csv(labs_path)
glucose_df = glucose_df[glucose_df['labname'] == 'bedside glucose']
glucose_df = glucose_df[glucose_df['patientunitstayid'].isin(used_stay_ids)]
glucose_df = glucose_df[['patientunitstayid', 'labresultoffset', 'labresult']]
glucose_df = glucose_df.rename(columns={'labresult': 'glucose', 
                                        'patientunitstayid': 'stay_id',
                                        'labresultoffset': 'observationoffset'})

# merge vital and glucose
vital_df = pd.merge(vital_df, glucose_df, 
                     on=['stay_id', 'observationoffset'], 
                     how='outer', 
                     suffixes=('_vital', '_glucose'))

# 对于vital列中不存在的值，将其他vital相关列设为NaN；对于glucose列不存在的值，将glucose列填充为NaN
vital_df['glucose'] = vital_df['glucose'].where(~vital_df['glucose'].isna(), np.nan)

# 仅保留前24小时内的数据
vital_df = vital_df[vital_df['observationoffset'] <= 24*60] 
vital_df = vital_df.sort_values(by=['stay_id', 'observationoffset'])

# get the timepoint for each feature
vital_columns = ['heartrate', 'systemicsystolic', 'systemicdiastolic', 'systemicmean','respiration', 'sao2', 'temperature', 'glucose']
vital_df['charttime'] = vital_df['observationoffset'].astype('int')
vital_df["timepoint"] = (vital_df['charttime'] // time_resolution_min).astype('int')

print(vital_df.head())

In [ ]:
# -----------------------------
# 1) Column setup
# -----------------------------
numeric_cols = vital_df.select_dtypes(include=[np.number]).columns.tolist()
id_cols = ['timepoint']
feature_cols = [c for c in numeric_cols if c not in id_cols]

for aux_col in ('subject_id', 'charttime'):
    if aux_col in feature_cols:
        feature_cols.remove(aux_col)

# -----------------------------
# 2) Define custom funcs
# -----------------------------
def _cv(x: pd.Series):
    m = x.mean()
    return np.nan if m == 0 or np.isnan(m) else x.std(ddof=1) / m

# -----------------------------
# 3) Groupby aggregate once
# -----------------------------
g = vital_df.groupby(id_cols, sort=False, observed=True)

agg_dict = {
    col: [('mean', 'mean'),
          ('std', 'std'),
          ('min', 'min'),
          ('max', 'max'),
          ('count', 'count'),
          ('cv', _cv)]
    for col in feature_cols
}

if 'subject_id' in vital_df.columns:
    agg_dict['subject_id'] = [('first', 'first')]
if 'charttime' in vital_df.columns:
    agg_dict['charttime'] = [('mean', 'mean')]

agg = g.agg(agg_dict)  # MultiIndex columns: (feature, stat)

# -----------------------------
# 4) Derive global means from the aggregated result
# -----------------------------
# Select all (feature, 'std') and (feature, 'cv') columns and compute row-wise mean
std_cols = [col for col in agg.columns if col[1] == 'std']
cv_cols  = [col for col in agg.columns if col[1] == 'cv']

# Compute per-group global means (skip NaN)
global_std_mean = agg.loc[:, std_cols].mean(axis=1, skipna=True)
global_cv_mean  = agg.loc[:, cv_cols].mean(axis=1, skipna=True)

# Attach as new single-level columns on the same index
agg[('GLOBAL', 'all_features_std_mean')] = global_std_mean
agg[('GLOBAL', 'all_features_cv_mean')]  = global_cv_mean

# -----------------------------
# 5) Flatten columns and finalize
# -----------------------------
out = agg.reset_index()
out.columns = [f"{a}_{b}" if b else f"{a}" for a, b in out.columns]

# Drop charttime aggregation if not needed
drop_cols = [c for c in out.columns if c.startswith('charttime_')]
if drop_cols:
    out.drop(columns=drop_cols, inplace=True)

# -----------------------------
# 6) Save
# -----------------------------
out_path = os.path.join(timeseries_dir, 'vital_PerTimepoint_stats.csv')
out.to_csv(out_path, index=False)


In [ ]:
# Convert charttime to timepoint intervals
obs_interval = 5 # 5 min
vital_df['obspoint'] = (vital_df['charttime'] // obs_interval).astype('int')
aggregated_vitals = vital_df.groupby(['stay_id', 'obspoint'])[vital_columns].agg(['min', 'max', 'mean']).reset_index()

aggregated_vitals.columns = ['_'.join(col).strip('_') for col in aggregated_vitals.columns]

# Rename stay_id and timepoint columns for clarity
aggregated_vitals.rename(columns={'stay_id_': 'stay_id', 'obspoint_': 'obspoint'}, inplace=True)

# Display the aggregated result
print(aggregated_vitals.head())

In [ ]:
# Change chartime to timepoints
final_vitals = aggregated_vitals.copy()
final_vitals['timepoint'] = (final_vitals['obspoint'] * obs_interval) // time_resolution_min
final_vitals.drop(columns=['obspoint'], inplace=True)

# Aggregate again by stay_id and timepoint, calculating the mean for all columns
final_vital = final_vitals.groupby(['stay_id', 'timepoint']).mean().reset_index()
final_vital.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col for col in final_vital.columns]
final_vital = final_vital.sort_values(by=['stay_id', 'timepoint'])

# Save final_vital to a CSV file
final_vital.to_csv(vital_export_path, index=False)

# Display the first few rows of the final result
print(final_vital.head())


## Lab Results

In [ ]:
lab_path = os.path.join(raw_dir, 'labs.csv')
lab_export_path = os.path.join(timeseries_dir, 'labs.csv')

labs = [
    'albumin', 'anion gap', 'Hct', 'Hgb', 'HCO3', 'total bilirubin', 'creatinine', 'chloride', 'glucose', 
    'BUN', 'PT', 'PTT', 'sodium', 'lactate', '-bands', 
    'potassium', 'platelets x 1000', 'WBC x 1000', 'PT-INR'
]

raw_lab_df = pd.read_csv(lab_path)
raw_lab_df = raw_lab_df[raw_lab_df['labname'].isin(labs)]
raw_lab_df = raw_lab_df.rename(columns={'patientunitstayid': 'stay_id'})
raw_lab_df = raw_lab_df[raw_lab_df['stay_id'].isin(used_stay_ids)]
raw_lab_df = raw_lab_df[raw_lab_df['labresultoffset'] < 24*60] # only keep the first 24 hours

raw_lab_df['charttime'] = raw_lab_df['labresultoffset']
raw_lab_df['timepoint'] = raw_lab_df['charttime']//time_resolution_min  # time resolution is 1 hour
raw_lab_df['timepoint'].astype('int')
raw_lab_df.drop(columns=['labresultoffset', 'charttime'], inplace=True)

raw_lab_df.info()

In [ ]:
# -----------------------------
# labs: per-(stay_id,timepoint) summary table
# output style consistent with vital_PerTimepoint_stats.csv
# -----------------------------
# 1) define coefficient of variation (CV)
def _cv(x: pd.Series):
    m = x.mean()
    return np.nan if m == 0 or np.isnan(m) else x.std(ddof=1) / m

# 2) group by (stay_id, timepoint, labname) and aggregate labresult
#    use observed=True to avoid Cartesian expansion of categorical groups
g = raw_lab_df.groupby(['timepoint', 'labname'], sort=False, observed=True)['labresult']

agg = g.agg(
    mean='mean',
    std=lambda s: s.std(ddof=1),
    min='min',
    max='max',
    count='count',
    cv=_cv
)

# 3) pivot labname from row index to columns, multi-level columns (stat, labname)
#    then swap levels to match vital code: (feature=labname, stat)
agg = agg.unstack('labname')                  # columns: (stat, labname)
agg = agg.swaplevel(0, 1, axis=1)             # columns: (labname, stat)
agg = agg.sort_index(axis=1)                  # optional: tidy column grouping

# 4) compute global means across all lab features for std/cv
std_cols = [col for col in agg.columns if col[1] == 'std']
cv_cols  = [col for col in agg.columns if col[1] == 'cv']

global_std_mean = agg.loc[:, std_cols].mean(axis=1, skipna=True)
global_cv_mean  = agg.loc[:, cv_cols].mean(axis=1, skipna=True)

agg[('GLOBAL', 'all_features_std_mean')] = global_std_mean
agg[('GLOBAL', 'all_features_cv_mean')]  = global_cv_mean

# 5) flatten column names and finalize
out = agg.reset_index()

# consistent with vital code: (feature, stat) -> "feature_stat"
# note: after reset_index, index columns appear as ('stay_id',''), ('timepoint',''), flatten to single level
out.columns = [f"{a}_{b}" if b else f"{a}" for a, b in out.columns]

# labs table has no charttime aggregation columns, so no need to drop; if added later, reuse same logic
# drop_cols = [c for c in out.columns if c.startswith('charttime_')]
# if drop_cols:
#     out.drop(columns=drop_cols, inplace=True)

# 6) save
lab_stats_path = os.path.join(timeseries_dir, 'lab_PerTimepoint_stats.csv')
out.to_csv(lab_stats_path, index=False)


In [ ]:
lab_df = raw_lab_df.pivot_table(
    index=['stay_id', 'timepoint'], 
    columns='labname', 
    values='labresult', 
    aggfunc='mean'
).reset_index()

lab_df.to_csv(lab_export_path, index=False)
lab_df.info()

# Merge vital signs and lab results according to the stay_id and timepoint

In [ ]:
demo_df = pd.read_csv(os.path.join(timeseries_dir, 'demographics.csv'))
vital_df = pd.read_csv(os.path.join(timeseries_dir, 'vital.csv'))
lab_df = pd.read_csv(os.path.join(timeseries_dir, 'labs.csv'))
subject_ids_df = demo_df[['subject_id', 'stay_id']].drop_duplicates()

ts_df = pd.merge(vital_df, lab_df, on=['stay_id', 'timepoint'], how='outer')
ts_df = ts_df.merge(subject_ids_df, on='stay_id', how='left')
ts_df = ts_df[ts_df['stay_id'].isin(used_stay_ids)]

included_stay_id = ts_df['stay_id'].unique()
demo_df = demo_df[demo_df['stay_id'].isin(included_stay_id)]
assert demo_df['stay_id'].unique().shape[0] == ts_df['stay_id'].unique().shape[0]
index_columns = ['subject_id', 'stay_id', 'timepoint']
other_columns = [col for col in ts_df.columns if col not in index_columns]
ts_df = ts_df[index_columns + other_columns]
ts_df = ts_df.sort_values(by=index_columns)

demo_df.to_csv(demo_export_path, index=False, sep=",")
print(ts_df["timepoint"].value_counts())

In [ ]:
# fill empty rows
subject_stay_df = ts_df[['subject_id', 'stay_id']].drop_duplicates().dropna()

# Generate timepoints
def generate_timepoints(row, n=num_timestep):
    return pd.DataFrame({
        'subject_id': [row['subject_id']] * n,
        'stay_id': [row['stay_id']] * n,
        'timepoint': list(range(n))})

firstn_timepoints = pd.concat(subject_stay_df.apply(generate_timepoints, axis=1).tolist()).reset_index(drop=True)

ts_df = ts_df.merge(firstn_timepoints, on=['subject_id', 'stay_id', 'timepoint'], how='outer')

# 填补所有 NaN 值（其他列）
for col in ts_df.columns:
    if col not in ['stay_id', 'timepoint', 'subject_id']:
        ts_df[col] = ts_df[col].fillna(np.nan)

# 保存结果
ts_df.to_csv(os.path.join(timeseries_dir, 'time-series.csv'), index=False, sep=',')

# 查看时间点分布
print(ts_df["timepoint"].value_counts())

## Labels
Includes the ICU-mortality, remaining length of stay, and medication

In [ ]:
# ICU mortality
mortality_df = pd.read_csv(os.path.join(raw_dir, 'label_icumortality.csv'))
mortality_export_csv_path = os.path.join(timeseries_dir, 'label_icumortality.csv')

mortality_df = mortality_df[mortality_df['stay_id'].isin(used_stay_ids)]
mortality_df.dropna(subset=['icu_mortality'], inplace=True)

used_stay_ids = mortality_df['stay_id'].unique()

mortality_df.to_csv(mortality_export_csv_path, index=False)
mortality_df.info()

In [ ]:
# Remaning ICU LOS
los_df = pd.read_csv(os.path.join(raw_dir, 'label_los.csv'))
los_export_csv_path = os.path.join(timeseries_dir, 'label_los.csv')

los_df = los_df[los_df['stay_id'].isin(used_stay_ids)]
los_df.dropna(subset=['remaining_los'], inplace=True)
used_stay_ids = los_df['stay_id'].unique()

los_df.to_csv(los_export_csv_path, index=False)
los_df.info()

In [ ]:
# ICU Phenotyping
import json
from utils.constants import phenotypes_ccs
phenotype_data_csv_path = os.path.join(raw_dir, 'diag_after_24h.csv')
phenotype_export_csv_path = os.path.join(timeseries_dir, 'label_phenotype.csv')
diag_after_24h_df = pd.read_csv(phenotype_data_csv_path , dtype={'icd9_code': str})
diag_after_24h_df = diag_after_24h_df[diag_after_24h_df['stay_id'].isin(used_stay_ids)]

# get the stay_id: [icd9_code1, icd9_code2, xxx] dict
stay_id_icd9_dict = {}
for stay_id in diag_after_24h_df['stay_id'].unique():
    icd9_codes = diag_after_24h_df[diag_after_24h_df['stay_id'] == stay_id]['icd9_code'].tolist()
    # remove '.' within icd9_code
    icd9_codes = [code.replace('.', '') for code in icd9_codes]
    stay_id_icd9_dict[stay_id] = icd9_codes

# map icd9_code to ccs function
def map_icd9_to_ccs_fn(icd9_code):
    for phenotype, icd9_list in phenotypes_ccs.items():
        if icd9_code in icd9_list:
            return phenotype
    return None

# map stay_id to ccs
stay_id_phenotype_dict = {}
for stay_id, icd9_codes in stay_id_icd9_dict.items():
    phenotypes = set()
    for icd9_code in icd9_codes:
        phenotype = map_icd9_to_ccs_fn(icd9_code)
        if phenotype is not None:
            phenotypes.add(phenotype)
    stay_id_phenotype_dict[stay_id] = json.dumps(list(phenotypes))

# create a dataframe
phenotype_df = pd.DataFrame(stay_id_phenotype_dict.items(), columns=['stay_id', 'phenotypes'])
phenotype_df.to_csv(phenotype_export_csv_path, index=False)
phenotype_df.info()

In [ ]:
# Medication
med_df = pd.read_csv(os.path.join(raw_dir, 'label_medication.csv'))
med_export_path = os.path.join(timeseries_dir, 'label_medication.csv')

med_df.rename(columns={'patientunitstayid': 'stay_id'}, inplace=True)
med_df = med_df[med_df['stay_id'].isin(used_stay_ids)]
med_df = med_df[['stay_id', 'drugname']]
med_df = med_df.drop_duplicates()

# Count occurrences of drugname
drug_counts = med_df['drugname'].value_counts()

# Calculate total occurrences of top 400 drugnames
top_400_drugs_count = drug_counts.head(400).sum()

# Calculate total occurrences of all drugnames
total_drugs_count = drug_counts.sum()

# Calculate ratio of top 100 drugnames
top_400_drugs_ratio = (top_400_drugs_count / total_drugs_count) * 100

# Output results
print(f"Total medication count: {total_drugs_count}")
print(f"Top 400 most common medications count: {top_400_drugs_count}")
print(f"Top 400 medications account for {top_400_drugs_ratio:.2f}% of total")


In [ ]:
# Keep the top 400 most common drugnames
top400_drug_names = drug_counts.head(400).index.tolist()

med_df = med_df[med_df['drugname'].isin(top400_drug_names)]
med_df['drugname_category'] = med_df['drugname'].astype('category').cat.codes

#Check processed data
print(med_df.head())

# Save processed data
med_df.to_csv(med_export_path, index=False)

In [ ]:
category_mapping = dict(enumerate(med_df['drugname'].astype('category').cat.categories))
print(category_mapping)

## Apply the used stay_ids

In [ ]:
demo_df = pd.read_csv(os.path.join(timeseries_dir, 'demographics.csv'))
ts_df = pd.read_csv(os.path.join(timeseries_dir, 'time-series.csv'))
diag_df = pd.read_csv(os.path.join(timeseries_dir, 'diag.csv'))
label_icumortality_df = pd.read_csv(os.path.join(timeseries_dir, 'label_icumortality.csv'))
label_los_df = pd.read_csv(os.path.join(timeseries_dir, 'label_los.csv'))
label_medication_df = pd.read_csv(os.path.join(timeseries_dir, 'label_medication.csv'))
label_phenotype_df = pd.read_csv(os.path.join(timeseries_dir, 'label_phenotype.csv'))

# apply used_stay_ids
demo_df = demo_df[demo_df['stay_id'].isin(used_stay_ids)]
ts_df = ts_df[ts_df['stay_id'].isin(used_stay_ids)]
diag_df = diag_df[diag_df['stay_id'].isin(used_stay_ids)]
label_icumortality_df = label_icumortality_df[label_icumortality_df['stay_id'].isin(used_stay_ids)]
label_los_df = label_los_df[label_los_df['stay_id'].isin(used_stay_ids)]
label_medication_df = label_medication_df[label_medication_df['stay_id'].isin(used_stay_ids)]
label_phenotype_df = label_phenotype_df[label_phenotype_df['stay_id'].isin(used_stay_ids)]

# add subject_id to label_dfs
stay_id_to_subject_id = demo_df[['stay_id', 'subject_id']].drop_duplicates().set_index('stay_id')
label_icumortality_df['subject_id'] = label_icumortality_df['stay_id'].map(stay_id_to_subject_id['subject_id'])
label_los_df['subject_id'] = label_los_df['stay_id'].map(stay_id_to_subject_id['subject_id'])
label_medication_df['subject_id'] = label_medication_df['stay_id'].map(stay_id_to_subject_id['subject_id'])
label_phenotype_df['subject_id'] = label_phenotype_df['stay_id'].map(stay_id_to_subject_id['subject_id'])

# export to csv
demo_df.to_csv(os.path.join(timeseries_dir, 'demographics.csv'), index=False)
ts_df.to_csv(os.path.join(timeseries_dir, 'time-series.csv'), index=False)
diag_df.to_csv(os.path.join(timeseries_dir, 'diag.csv'), index=False)
label_icumortality_df.to_csv(os.path.join(timeseries_dir, 'label_icumortality.csv'), index=False)
label_los_df.to_csv(os.path.join(timeseries_dir, 'label_los.csv'), index=False)
label_medication_df.to_csv(os.path.join(timeseries_dir, 'label_medication.csv'), index=False)
label_phenotype_df.to_csv(os.path.join(timeseries_dir, 'label_phenotype.csv'), index=False)